<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/09)_%EC%9B%90%ED%95%98%EB%8A%94_txt_%ED%8C%8C%EC%9D%BC_%EC%97%85%EB%A1%9C%EB%93%9C_%EB%B0%8F_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 연결

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 프로젝트 루트로 이동

%cd /content/drive/MyDrive/rag_intent_chatbot

# 확인

!pwd

/content/drive/MyDrive/rag_intent_chatbot
/content/drive/MyDrive/rag_intent_chatbot


In [ ]:
# 필수 파일 존재 여부 확인

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/rag_intent_chatbot")

required_paths = [
    PROJECT_ROOT / "models" / "intent_classifier.pt",
    PROJECT_ROOT / "data" / "intents.json",
    PROJECT_ROOT / "src" / "intent" / "predict.py",
    PROJECT_ROOT / "src" / "rag" / "document_loader.py",
    PROJECT_ROOT / "src" / "rag" / "text_preprocessor.py",
    PROJECT_ROOT / "src" / "rag" / "chunker.py",
    PROJECT_ROOT / "src" / "rag" / "embedder.py",
    PROJECT_ROOT / "src" / "rag" / "vector_store.py",
    PROJECT_ROOT / "src" / "rag" / "retriever.py",
    PROJECT_ROOT / "src" / "rag" / "answer_generator.py",
    PROJECT_ROOT / "src" / "chatbot.py",
]

all_exists = True

for path in required_paths:
    exists = path.exists()
    print(f"{'[OK]' if exists else '[없음]'} {path.relative_to(PROJECT_ROOT)}")

    if not exists:
        all_exists = False

if all_exists:
    print("\n필수 파일 확인 완료")
else:
    print("\n일부 필수 파일이 없습니다.")

[OK] models/intent_classifier.pt
[OK] data/intents.json
[OK] src/intent/predict.py
[OK] src/rag/document_loader.py
[OK] src/rag/text_preprocessor.py
[OK] src/rag/chunker.py
[OK] src/rag/embedder.py
[OK] src/rag/vector_store.py
[OK] src/rag/retriever.py
[OK] src/rag/answer_generator.py
[OK] src/chatbot.py

필수 파일 확인 완료


In [ ]:
# 필수 라이브러리 설치 및 import
# google genai 2.18.0 버전 사용

#설치
!pip install -q -r requirements.txt
!pip install -q google-genai==2.18.0 google-auth==2.49.0 # colab 환경에서는 2.49.0 버전을 추천하기에(google-auth==2.49.0는 없애도 지장은 없음)

# import

import os
import shutil
from pathlib import Path
from importlib.metadata import version

from google.colab import files, userdata
from google import genai

from src.intent.predict import IntentPredictor

from src.rag.document_loader import Document, load_documents
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_documents
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever
from src.rag.answer_generator import AnswerGenerator

from src.chatbot import Chatbot, ChatbotResult

print("google-genai:", version("google-genai"))
print("import 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.6 MB/s eta 0:00:00
ERROR: Cannot install google-auth==2.49.0 and google-genai==2.18.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
google-genai: 2.12.1
import 완료


In [ ]:
# 구글 API 키 확인

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "Colab Secrets에서 GEMINI_API_KEY를 찾을 수 없습니다."
    )

print("GEMINI_API_KEY 확인 완료")

GEMINI_API_KEY 확인 완료


In [ ]:
# 업로드 폴더 생성
# 기존의 data/documents/ 구조에서 data/uploaded_documents/ 구조로 바꿔 진행

UPLOAD_DIRECTORY = PROJECT_ROOT / "data" / "uploaded_documents"
STAGING_DIRECTORY = Path("/content/rag_upload_staging")

UPLOAD_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("사용자 업로드 폴더:")
print(UPLOAD_DIRECTORY)

사용자 업로드 폴더:
/content/drive/MyDrive/rag_intent_chatbot/data/uploaded_documents


In [ ]:
# 기존 data/uploaded_documents/ 내의 업로드된 파일들을 정리한다

CLEAR_PREVIOUS_UPLOADS = True

if CLEAR_PREVIOUS_UPLOADS:
    for path in UPLOAD_DIRECTORY.iterdir():
        if path.is_file():
            path.unlink()

    print("기존 uploaded_documents 파일을 정리했습니다.")

else:
    print("기존 uploaded_documents 파일을 유지합니다.")

기존 uploaded_documents 파일을 정리했습니다.


In [ ]:
# pc 내에서 파일을 선택
# 여러 파일을 선택할 수도 있다.

def select_files():
    if STAGING_DIRECTORY.exists():
        shutil.rmtree(STAGING_DIRECTORY)

    STAGING_DIRECTORY.mkdir(parents=True, exist_ok=True)

    original_directory = Path.cwd()

    try:
        os.chdir(STAGING_DIRECTORY)
        uploaded_files = files.upload()

    finally:
        os.chdir(original_directory)

    return uploaded_files


uploaded = select_files()

print()
print("선택된 파일 수:", len(uploaded))

In [ ]:
# txt 파일 검사 및 저장

def save_valid_txt_files(uploaded_files):
    saved_files = []

    for filename, file_bytes in uploaded_files.items():

        file_name_only = Path(filename).name
        suffix = Path(file_name_only).suffix.lower()

        if suffix != ".txt":
            print()
            print("[업로드 제외]")
            print(f"파일명: {file_name_only}")
            print("이 단계에서는 .txt 파일만 지원합니다.")
            continue

        save_path = UPLOAD_DIRECTORY / file_name_only

        save_path.write_bytes(file_bytes)

        saved_files.append(save_path)

        print()
        print("[업로드 완료]")
        print(f"파일명: {file_name_only}")
        print(f"크기: {len(file_bytes)} bytes")
        print(
            "저장 위치:",
            save_path.relative_to(PROJECT_ROOT)
        )
        print("저장 성공 여부:", save_path.exists())

    if STAGING_DIRECTORY.exists():
        shutil.rmtree(STAGING_DIRECTORY)

    return saved_files


saved_files = save_valid_txt_files(uploaded)

if not saved_files:
    print()
    print("사용 가능한 txt 파일이 없습니다.")
    print("먼저 txt 파일을 업로드해주세요.")

In [ ]:
# 저장된 txt 파일 확인

txt_files = sorted(UPLOAD_DIRECTORY.glob("*.txt"))

if not txt_files:
    print("사용 가능한 txt 파일이 없습니다.")
    print("먼저 txt 파일을 업로드해주세요.")

else:
    print(f"사용 가능한 txt 파일 수: {len(txt_files)}")

    for index, path in enumerate(txt_files, start=1):
        print(
            f"{index}. {path.name} "
            f"({path.stat().st_size} bytes)"
        )

In [ ]:
# 업로드한 문서 전용 Retriever 생성

# Retriever 생성 함수
# 함수의 구조는 기존의 main.py와 동일한 흐름이다.(기존의 main.py에서 build_retriever을 이용해 retriever을 만들었었음)

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
RETRIEVAL_TOP_K = 3


def build_uploaded_document_retriever(
    document_directory: str,
) -> Retriever:

    documents = load_documents(
        directory_path=document_directory
    )

    if not documents:
        raise ValueError(
            "불러올 수 있는 문서가 없습니다."
        )

    preprocessed_documents = []

    for document in documents:
        processed_text = preprocess_text(
            document.text
        )

        processed_document = Document(
            text=processed_text,
            metadata=document.metadata,
        )

        preprocessed_documents.append(
            processed_document
        )

    chunks = chunk_documents(
        documents=preprocessed_documents,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    if not chunks:
        raise ValueError(
            "생성된 Chunk가 없습니다."
        )

    print(f"불러온 문서 수: {len(documents)}")
    print(f"생성된 Chunk 수: {len(chunks)}")

    embedder = TextEmbedder()

    chunk_texts = [
        chunk.text
        for chunk in chunks
    ]

    chunk_embeddings = embedder.encode_texts(
        texts=chunk_texts
    )

    vector_store = VectorStore()

    vector_store.add(
        chunks=chunks,
        embeddings=chunk_embeddings,
    )

    retriever = Retriever(
        embedder=embedder,
        vector_store=vector_store,
    )

    return retriever

# Retriever 생성

txt_files = sorted(
    UPLOAD_DIRECTORY.glob("*.txt")
)

if not txt_files:
    retriever = None

    print("사용 가능한 txt 파일이 없습니다.")
    print("먼저 txt 파일을 업로드해주세요.")

else:
    retriever = build_uploaded_document_retriever(
        document_directory=str(UPLOAD_DIRECTORY)
    )

    print()
    print("업로드 문서 전용 Retriever 생성 완료")

In [ ]:
# IntentPredictor 생성
# IntentClassifier은 기존에 이미 학습을 진행하였다

MODEL_PATH = "models/intent_classifier.pt"
CONFIDENCE_THRESHOLD = 0.60

intent_predictor = IntentPredictor(
    model_path=MODEL_PATH
)

print("IntentPredictor 생성 완료")

In [ ]:
# Gemini Client 생성
# gemini-3.5-flash_lite 사용

gemini_client = genai.Client(
    api_key=api_key
)

print("Gemini client 생성 완료")

In [ ]:
# AnswerGenerator 생성
# 기존 인터페이스를 그대로 사용한다

answer_generator = AnswerGenerator(
    client=gemini_client,
    model_name="gemini-3.5-flash-lite",
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
    temperature=0.2,
    max_output_tokens=500,
)

print("AnswerGenerator 생성 완료")

In [ ]:
# Chatbot 생성

if retriever is None:
    chatbot = None

    print(
        "Retriever가 없어서 Chatbot을 "
        "생성할 수 없습니다."
    )

else:
    chatbot = Chatbot(
        intent_predictor=intent_predictor,
        retriever=retriever,
        answer_generator=answer_generator,
        confidence_threshold=0.60,
        retrieval_top_k=3,
        preview_length=200,
    )

    print("Chatbot 생성 완료")

In [ ]:
# Retriever 단독 테스트

if retriever is None:
    print("Retriever가 없습니다.")

else:
    retrieval_test_question = input(
        "검색 테스트 질문: "
    ).strip()

    search_results = retriever.retrieve(
        query=retrieval_test_question,
        top_k=RETRIEVAL_TOP_K,
    )

    print()
    print("질문:", retrieval_test_question)
    print(
        "검색 결과 개수:",
        len(search_results)
    )

    for result in search_results:
        print()
        print(f"[Rank {result.rank}]")
        print(f"score: {result.score:.4f}")
        print(
            "source:",
            result.chunk.source
        )
        print(
            "chunk_id:",
            result.chunk.chunk_id
        )
        print(
            "text:",
            result.chunk.text[:300]
        )

In [ ]:
# LLM 문서 답변 단독 테스트

if retriever is None:
    print("Retriever가 없습니다.")

else:
    answer_test_question = input(
        "LLM 답변 테스트 질문: "
    ).strip()

    answer_search_results = retriever.retrieve(
        query=answer_test_question,
        top_k=RETRIEVAL_TOP_K,
    )

    generated_answer = answer_generator.generate(
        question=answer_test_question,
        search_results=answer_search_results,
    )

    print()
    print("질문:")
    print(answer_test_question)

    print()
    print("[사용한 Context]")
    print(generated_answer.context)

    print()
    print("[최종 Answer]")
    print(generated_answer.answer)

    print()
    print(
        "has_relevant_context:",
        generated_answer.has_relevant_context
    )

    print(
        "사용한 Chunk 개수:",
        len(generated_answer.used_results)
    )

    print(
        "sources:",
        generated_answer.sources
    )

In [ ]:
# 전체 Chatbot 테스트

if chatbot is None:
    print("Chatbot이 없습니다.")

else:
    chatbot_test_input = input(
        "Chatbot 테스트 입력: "
    ).strip()

    result = chatbot.process_message(
        user_input=chatbot_test_input
    )

    print()
    print("사용자 입력:")
    print(result.user_input)

    print()
    print(
        "predicted_intent:",
        result.predicted_intent
    )

    print(
        "confidence:",
        f"{result.confidence:.4f}"
    )

    print(
        "fallback 여부:",
        result.is_fallback
    )

    print(
        "requires_rag:",
        result.requires_rag
    )

    print(
        "검색 결과 개수:",
        len(result.search_results)
    )

    print()
    print("[최종 response]")
    print(result.response)

In [ ]:
# 실제 대화 반복

def is_exit_command(text: str) -> bool:
    return text.strip().lower() in {
        "exit",
        "quit",
        "종료",
    }


if chatbot is None:
    print("Chatbot이 없습니다.")

else:
    print("업로드 문서 RAG 챗봇")
    print(
        "종료하려면 "
        "exit / quit / 종료 입력"
    )

    while True:
        user_input = input("\n사용자: ").strip()

        if is_exit_command(user_input):
            print("챗봇을 종료합니다.")
            break

        if not user_input:
            continue

        result = chatbot.process_message(
            user_input=user_input
        )

        print()
        print("챗봇:")
        print(result.response)

        print()
        print(
            f"Intent: "
            f"{result.predicted_intent}"
        )

        print(
            f"Confidence: "
            f"{result.confidence:.4f}"
        )

        if result.search_results:
            print()
            print("[검색 결과]")

            for search_result in result.search_results:
                print(
                    f"- rank="
                    f"{search_result.rank}, "
                    f"score="
                    f"{search_result.score:.4f}, "
                    f"source="
                    f"{search_result.chunk.source}, "
                    f"chunk_id="
                    f"{search_result.chunk.chunk_id}"
                )

업로드 문서 RAG 챗봇
종료하려면 exit / quit / 종료 입력
챗봇을 종료합니다.


In [ ]:
'''
실제 테스트 결과(한국외대 26-1학기 수강편람(서울캠퍼스)를 사용하였으며, 질문은 "졸업 학점은 몇이야?" 이였다.) Intent Classifier이 질문의 의도를 파악하는 과정에서 잘 파악하지 못하는 점을 발견하였다.
예상되는 이유는 Chatbot 용 Intent Classifier를 만드는 과정에서 사용자의 의도를 분류할 때에, 문서에 관한 질문은 "문서", "자료", "파일"을 일반적으로 포함하고 있으며, 해당하는 내용이 없을 시에는 질문의 의도를 파악하지 못하는 것이였음으로 예상된다.
또한, 의도 분석이 사용자가 이러한 질문을 왜 하였느냐가 아닌, 어떠한 질문을 하였는가이기에 해당하는 부분의 보완이 필요하다.
'''

'\n실제 테스트 결과(한국외대 26-1학기 수강편람(서울캠퍼스)를 사용하였으며, 질문은 "졸업 학점은 몇이야?" 이였다.) Intent Classifier이 질문의 의도를 파악하는 과정에서 잘 파악하지 못하는 점을 발견하였다.\n예상되는 이유는 Chatbot 용 Intent Classifier를 만드는 과정에서 사용자의 의도를 분류할 때에, 문서에 관한 질문은 "문서", "자료", "파일"을 일반적으로 포함하고 있으며, 해당하는 내용이 없을 시에는 질문의 의도를 파악하지 못하는 것이였음으로 예상된다.\n또한, 의도 분석이 사용자가 이러한 질문을 왜 하였느냐가 아닌, 어떠한 질문을 하였는가이기에 해당하는 부분의 보완이 필요하다.\n'

In [ ]:
# 새로운 문서로 교체

for path in UPLOAD_DIRECTORY.iterdir():
    if path.is_file():
        path.unlink()

print("기존 업로드 문서를 삭제했습니다.")

new_uploaded = select_files()

new_saved_files = save_valid_txt_files(
    new_uploaded
)

if not new_saved_files:
    retriever = None
    chatbot = None

    print()
    print("사용 가능한 txt 파일이 없습니다.")

else:
    retriever = build_uploaded_document_retriever(
        document_directory=str(
            UPLOAD_DIRECTORY
        )
    )

    chatbot = Chatbot(
        intent_predictor=intent_predictor,
        retriever=retriever,
        answer_generator=answer_generator,
        confidence_threshold=0.60,
        retrieval_top_k=3,
        preview_length=200,
    )

    print()
    print("새 문서용 Retriever 재생성 완료")
    print("새 문서용 Chatbot 재생성 완료")